# Part 1 - CNN Architectures (IR / Thermal)

Experiment with 4 CNN architectures of increasing complexity on IR patches.  
Each architecture has >= 10x the parameters of its predecessor.

**Requirements:**
- At least 4 architectures, first simple, each successive >= 10x params
- All trained >= 50 epochs with early stopping (patience=10)
- Generate model summaries
- ONE figure with 4 plots: params (log x) vs train/val accuracy & F1
- Identify optimal architecture

In [ ]:
import torch
from torchinfo import summary
from models import build_cnn, verify_10x_rule
from training import get_dataloaders, train_model
from analysis import plot_4panel, plot_training_curves, print_results_table

MANIFEST = "manifests/ir_manifest.csv"
MODALITY = "IR"
BATCH_SIZE = 256
EPOCHS = 70  # max epochs (early stop after 50 with patience 10)
LR = 1e-3
MIN_EPOCHS = 50
PATIENCE = 10

print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

## 1. Verify Parameter Growth Rule

In [ ]:
param_counts = verify_10x_rule()

## 2. Model Summaries

In [ ]:
for arch in ["A", "B", "C", "D"]:
    model, count = build_cnn(arch)
    print(f"\n{'='*60}")
    print(f"Architecture {arch}: {count:,} parameters")
    print(f"{'='*60}")
    summary(model, input_size=(1, 3, 64, 64), verbose=1)
    del model

## 3. Load Data

In [ ]:
train_loader, val_loader, train_df, val_df = get_dataloaders(
    MANIFEST, batch_size=BATCH_SIZE, num_workers=4
)
print(f"Train: {len(train_loader.dataset):,} samples ({len(train_loader)} batches)")
print(f"Val:   {len(val_loader.dataset):,} samples ({len(val_loader)} batches)")

## 4. Train All 4 Architectures

In [ ]:
results = []
histories = {}

for arch in ["A", "B", "C", "D"]:
    print(f"\n{'#'*60}")
    print(f"# Training Architecture {arch} ({param_counts[arch]:,} params)")
    print(f"{'#'*60}")

    model, count = build_cnn(arch)
    history = train_model(
        model, train_loader, val_loader,
        epochs=EPOCHS, lr=LR,
        patience=PATIENCE, min_epochs=MIN_EPOCHS,
        checkpoint_name=f"part1_ir_arch{arch}"
    )
    histories[arch] = history

    results.append({
        "label": f"Arch {arch}",
        "arch": arch,
        "param_count": count,
        "best_epoch": history["best_epoch"],
        "best_train_acc": history["best_train_acc"],
        "best_val_acc": history["best_val_acc"],
        "best_train_f1": history["best_train_f1"],
        "best_val_f1": history["best_val_f1"],
    })

    del model
    torch.cuda.empty_cache()

## 5. Results Summary

In [ ]:
print_results_table(results)

## 6. Required 4-Panel Figure

In [ ]:
x_values = [r["param_count"] for r in results]

plot_4panel(
    results, x_values,
    x_label="Number of Parameters",
    title=f"Part 1 - {MODALITY}: CNN Architecture Comparison",
    x_log=True,
    save_name=f"part1_{MODALITY.lower()}_architectures.png"
)

## 7. Individual Training Curves

In [ ]:
for arch in ["A", "B", "C", "D"]:
    plot_training_curves(
        histories[arch],
        title=f"{MODALITY} Architecture {arch} ({param_counts[arch]:,} params)",
        save_name=f"part1_{MODALITY.lower()}_arch{arch}_curves.png"
    )

## 8. Identify Optimal Architecture

In [ ]:
best = max(results, key=lambda r: r["best_val_f1"])
print(f"\nOptimal {MODALITY} architecture: {best['label']}")
print(f"  Parameters: {best['param_count']:,}")
print(f"  Val Accuracy: {best['best_val_acc']:.4f}")
print(f"  Val F1: {best['best_val_f1']:.4f}")
print(f"  Best Epoch: {best['best_epoch']}")

# Identify model with modest overfitting (for Part 2)
print("\nOverfitting analysis (train - val gap):")
for r in results:
    gap_acc = r["best_train_acc"] - r["best_val_acc"]
    gap_f1 = r["best_train_f1"] - r["best_val_f1"]
    print(f"  {r['label']}: acc gap = {gap_acc:.4f}, F1 gap = {gap_f1:.4f}")

---
**Next:** Use the architecture with modest overfitting for Part 2 regularization experiments.